In [19]:
# importa as bilbiotecas

import requests as rq
import urllib3
import pandas as pd

In [20]:
#faz o request
# Evita warning de certificado SSL (para testes locais)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = 'https://api-comexstat.mdic.gov.br/cities'

headers = {
    'Accept': 'application/json',
    'Content-Type': 'application/json'
}

body = {
   "flow": "export",
   "monthDetail": True,
   "period":{
       "from": "2025-01",
       "to": "2025-12"
   },
   "filters":[
       {
           "filter":"heading",
           "values":[6401,6402,6403,6404,6405]
       }
   ],
   "details":[
       "country",
       "state",
       "heading",
       "city",
   ],
   "metrics":[
       "metricFOB",
       "metricKG"
   ]

}

response = rq.post(url, headers=headers, json=body, verify=False)

print("Status Code:", response.status_code)
print(response.text)


Status Code: 200
{"data":{"list":[{"noMunMinsgUf":"Igrejinha - RS","year":"2025","monthNumber":"02","country":"Argentina","state":"Rio Grande do Sul","headingCode":"6402","heading":"Outro cal\u00e7ado com sola exterior e parte superior de borracha ou pl\u00e1stico","metricFOB":"3692026","metricKG":"102216"},{"noMunMinsgUf":"Novo Hamburgo - RS","year":"2025","monthNumber":"01","country":"Estados Unidos","state":"Rio Grande do Sul","headingCode":"6403","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de couro natural","metricFOB":"3595136","metricKG":"81594"},{"noMunMinsgUf":"Dois Irm\u00e3os - RS","year":"2025","monthNumber":"02","country":"Estados Unidos","state":"Rio Grande do Sul","headingCode":"6403","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de couro natural","metricFOB":"3442490","metricKG":"62718"},{"noMunMinsgUf":"Franca - SP",

In [21]:
#transforma o retorno em json
json_dados = response.json()
print(json_dados)

{'data': {'list': [{'noMunMinsgUf': 'Igrejinha - RS', 'year': '2025', 'monthNumber': '02', 'country': 'Argentina', 'state': 'Rio Grande do Sul', 'headingCode': '6402', 'heading': 'Outro calçado com sola exterior e parte superior de borracha ou plástico', 'metricFOB': '3692026', 'metricKG': '102216'}, {'noMunMinsgUf': 'Novo Hamburgo - RS', 'year': '2025', 'monthNumber': '01', 'country': 'Estados Unidos', 'state': 'Rio Grande do Sul', 'headingCode': '6403', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de couro natural', 'metricFOB': '3595136', 'metricKG': '81594'}, {'noMunMinsgUf': 'Dois Irmãos - RS', 'year': '2025', 'monthNumber': '02', 'country': 'Estados Unidos', 'state': 'Rio Grande do Sul', 'headingCode': '6403', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de couro natural', 'metricFOB': '3442490', 'metricKG': '62718'}, {'noMunMinsgUf': 'Franca - SP', 'yea

In [22]:
#cria dataframe
normaliza_dados = pd.json_normalize(json_dados['data']['list'])
data_frame = pd.DataFrame(normaliza_dados)

In [23]:
#ajusta o tipo dos dados
data_frame = data_frame.astype({
    'year': int,
    'monthNumber': int,
    'metricFOB': float,
    'metricKG': float,
    'headingCode': int
})

In [24]:
#ordena os dados
data_frame = data_frame.sort_values(by='monthNumber', ascending=True)

In [25]:
#cria coluna dia
data_frame['dia'] = 1

In [26]:
#cria coluna data
data_frame['data'] = (data_frame['dia'].astype(str).str.zfill(2)+'/'+data_frame['monthNumber'].astype(str).str.zfill(2)+'/'+data_frame['year'].astype(str))

In [27]:
#ordena colunas
data_frame = data_frame[['year','monthNumber','dia','data','headingCode','heading','country','state','noMunMinsgUf','metricFOB','metricKG']]

In [29]:
#salva dados em excel
data_frame.to_excel('./dados_exportacao.xlsx', header=True, index=False,sheet_name='exportacao')